# 4 — Random Under-Sampling

Sweeps how many Non-SEP samples to keep in training, from a mild trim down to a
heavy 500. No synthetic data is involved — this isolates the effect of
rebalancing by discarding majority samples.

**Input:** `final_split_data_HybridNorm_Tomek` (from notebook 3)

**Output:** `results/<classifier>_tomek_nonsep_{8000,2000,500}.txt`

**Run after:** notebook 3.

## Variants

`tomek_nonsep_8000` · `tomek_nonsep_2000` · `tomek_nonsep_500` — the number is how many
Non-SEP training samples survive. SEP samples are always kept in full (118).

Only the training set is resampled; validation and test keep their natural
0.96% positive rate.


## 1. Load Tomek Data and Build Under-Sampled Variants

In [1]:
import pickle
import numpy as np

# ══════════════════════════════════════════════════════════════
# LOAD TOMEK-CLEANED HYBRID DATASET
# Then create Non-SEP undersampled datasets in memory only
# ══════════════════════════════════════════════════════════════

def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]


# ---------------------------------------------------------
# Load Tomek-cleaned dataset
# ---------------------------------------------------------
folder = "./final_split_data_HybridNorm_Tomek"

X_train, y_train = load_split(f"{folder}/train_set.pkl")
X_val,   y_val   = load_split(f"{folder}/val_set.pkl")
X_test,  y_test  = load_split(f"{folder}/test_set.pkl")

print("Loaded Tomek-cleaned dataset:")
print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape,   y_val.shape)
print("Test :", X_test.shape,  y_test.shape)


# ---------------------------------------------------------
# Create undersampled training variants
# Keep all SEP samples unchanged
# Only reduce Non-SEP samples
# ---------------------------------------------------------
nonsep_targets = [8000, 2000, 500]

datasets_undersampled = {}

# Class indices
nonsep_idx = np.where(y_train.astype(int) == 0)[0]
sep_idx    = np.where(y_train.astype(int) == 1)[0]

print("\nOriginal Tomek-cleaned training class counts:")
print("Non-SEP:", len(nonsep_idx))
print("SEP    :", len(sep_idx))
print(f"Ratio  : {len(nonsep_idx) / len(sep_idx):.1f}x")
print("─" * 70)

# Reproducibility
rng = np.random.default_rng(seed=42)

for target_nonsep in nonsep_targets:

    if target_nonsep > len(nonsep_idx):
        raise ValueError(
            f"Requested {target_nonsep} Non-SEP samples, but only "
            f"{len(nonsep_idx)} are available after Tomek."
        )

    # Randomly select Non-SEP samples
    selected_nonsep_idx = rng.choice(
        nonsep_idx,
        size=target_nonsep,
        replace=False
    )

    # Keep all SEP samples
    selected_idx = np.concatenate([selected_nonsep_idx, sep_idx])

    # Shuffle selected training samples after undersampling
    rng.shuffle(selected_idx)

    X_train_us = X_train[selected_idx].astype(np.float32)
    y_train_us = y_train[selected_idx]

    dataset_name = f"tomek_nonsep_{target_nonsep}"

    datasets_undersampled[dataset_name] = {
        "X_train": X_train_us,
        "y_train": y_train_us,

        "X_val": X_val,
        "y_val": y_val,

        "X_test": X_test,
        "y_test": y_test,

        "nonsep_kept": target_nonsep,
        "sep_kept": len(sep_idx),
    }

    counts = np.bincount(y_train_us.astype(int), minlength=2)
    ratio = counts[0] / counts[1]

    print(f"[{dataset_name}]")
    print("  Train shape:", X_train_us.shape)
    print("  Class 0 / Non-SEP:", counts[0])
    print("  Class 1 / SEP    :", counts[1])
    print(f"  Ratio            : {ratio:.1f}x")
    print("─" * 70)


# ---------------------------------------------------------
# Final summary
# ---------------------------------------------------------
print("\n✅ Undersampled Tomek datasets are ready in memory.")
print("Dictionary name: datasets_undersampled")
print("Datasets:", list(datasets_undersampled.keys()))

Loaded Tomek-cleaned dataset:
Train: (12441, 288, 10) (12441,)
Val  : (1780, 288, 10) (1780,)
Test : (3559, 288, 10) (3559,)

Original Tomek-cleaned training class counts:
Non-SEP: 12323
SEP    : 118
Ratio  : 104.4x
──────────────────────────────────────────────────────────────────────
[tomek_nonsep_8000]
  Train shape: (8118, 288, 10)
  Class 0 / Non-SEP: 8000
  Class 1 / SEP    : 118
  Ratio            : 67.8x
──────────────────────────────────────────────────────────────────────
[tomek_nonsep_2000]
  Train shape: (2118, 288, 10)
  Class 0 / Non-SEP: 2000
  Class 1 / SEP    : 118
  Ratio            : 16.9x
──────────────────────────────────────────────────────────────────────
[tomek_nonsep_500]
  Train shape: (618, 288, 10)
  Class 0 / Non-SEP: 500
  Class 1 / SEP    : 118
  Ratio            : 4.2x
──────────────────────────────────────────────────────────────────────

✅ Undersampled Tomek datasets are ready in memory.
Dictionary name: datasets_undersampled
Datasets: ['tomek_nonsep_8

## 2. Catch22 Feature Extraction (SVM Input)

In [6]:
import numpy as np
from joblib import Parallel, delayed
from sktime.transformations.panel.catch22 import Catch22
from sklearn.impute import SimpleImputer
import warnings

# ══════════════════════════════════════════════════════════════
# CATCH22 FEATURE EXTRACTION FOR TOMEK + UNDERSAMPLED DATASETS
# Input : datasets_undersampled
#         tomek_nonsep_8000, tomek_nonsep_2000, tomek_nonsep_500
# Output: datasets_svm_undersampled
# ══════════════════════════════════════════════════════════════

def _catch22_chunk(X_chunk):
    transformer = Catch22(catch24=False)
    return transformer.fit_transform(X_chunk).to_numpy()


def extract_catch22(X, label="", n_jobs=-1, chunk_size=200):

    n_samples, n_timepoints, n_channels = X.shape

    # Replace NaN / Inf before Catch22
    if not np.isfinite(X).all():
        n_bad = (~np.isfinite(X)).sum()
        print(f"    ⚠️ {n_bad} non-finite values in {label} — replacing with channel median")

        X = X.copy()

        for c in range(n_channels):
            col = X[:, :, c]
            median_c = np.nanmedian(col)

            if not np.isfinite(median_c):
                median_c = 0.0

            col[~np.isfinite(col)] = median_c
            X[:, :, c] = col

    # sktime expects shape: (samples, channels, timesteps)
    X_sktime = X.transpose(0, 2, 1).astype(np.float64)

    chunks = [
        X_sktime[i:i + chunk_size]
        for i in range(0, n_samples, chunk_size)
    ]

    print(f"    {label}: {n_samples} samples in {len(chunks)} chunks", flush=True)

    results = Parallel(n_jobs=n_jobs)(
        delayed(_catch22_chunk)(chunk)
        for chunk in chunks
    )

    X_feat = np.vstack(results)

    expected_cols = 22 * n_channels

    assert X_feat.shape == (n_samples, expected_cols), (
        f"Unexpected shape for {label}: {X_feat.shape}, "
        f"expected ({n_samples}, {expected_cols})"
    )

    return X_feat.astype(np.float32)


# ══════════════════════════════════════════════════════════════
# Sanity check before extraction
# ══════════════════════════════════════════════════════════════

print("Pre-extraction value range check:")
print(
    f"{'Dataset':<18} "
    f"{'Train shape':<22} "
    f"{'Finite?':<10} "
    f"{'Min':>12} "
    f"{'Max':>12} "
    f"{'NaN/Inf':>10} "
    f"{'Class 0':>8} "
    f"{'Class 1':>8} "
    f"{'Ratio':>8}"
)
print("─" * 120)

for dataset_name, d in datasets_undersampled.items():
    X = d["X_train"]
    y = d["y_train"]

    counts = np.bincount(y.astype(int), minlength=2)
    ratio = counts[0] / counts[1] if counts[1] > 0 else np.inf

    print(
        f"{dataset_name:<18} "
        f"{str(X.shape):<22} "
        f"{str(np.isfinite(X).all()):<10} "
        f"{np.nanmin(X):>12.4f} "
        f"{np.nanmax(X):>12.4f} "
        f"{(~np.isfinite(X)).sum():>10} "
        f"{counts[0]:>8} "
        f"{counts[1]:>8} "
        f"{ratio:>7.1f}x"
    )


# ══════════════════════════════════════════════════════════════
# Extract Catch22 features
# ══════════════════════════════════════════════════════════════

datasets_svm_undersampled = {}

print("\nExtracting Catch22 features for Tomek + undersampled datasets...")

for dataset_name, d in datasets_undersampled.items():

    print(f"\n[{dataset_name}] extracting train...")
    X_tr = extract_catch22(
        d["X_train"],
        label=f"{dataset_name}/train",
        n_jobs=-1,
        chunk_size=200
    )

    print(f"[{dataset_name}] extracting val...")
    X_va = extract_catch22(
        d["X_val"],
        label=f"{dataset_name}/val",
        n_jobs=-1,
        chunk_size=200
    )

    print(f"[{dataset_name}] extracting test...")
    X_te = extract_catch22(
        d["X_test"],
        label=f"{dataset_name}/test",
        n_jobs=-1,
        chunk_size=200
    )

    datasets_svm_undersampled[dataset_name] = {
        "X_train": X_tr,
        "y_train": d["y_train"],

        "X_val": X_va,
        "y_val": d["y_val"],

        "X_test": X_te,
        "y_test": d["y_test"],

        "nonsep_kept": d.get("nonsep_kept", None),
        "sep_kept": d.get("sep_kept", None),
    }

    print(
        f"[{dataset_name}] ✓ "
        f"train {X_tr.shape} | "
        f"val {X_va.shape} | "
        f"test {X_te.shape}"
    )


# ══════════════════════════════════════════════════════════════
# Impute Catch22 features
# Fit imputer on train only, apply to val/test
# ══════════════════════════════════════════════════════════════

print("\nChecking and imputing Catch22 features...")

for dataset_name in list(datasets_svm_undersampled.keys()):

    d = datasets_svm_undersampled[dataset_name]

    X_tr = d["X_train"].copy()
    X_va = d["X_val"].copy()
    X_te = d["X_test"].copy()

    y_tr = d["y_train"]
    y_va = d["y_val"]
    y_te = d["y_test"]

    n_bad_tr = (~np.isfinite(X_tr)).sum()
    n_bad_va = (~np.isfinite(X_va)).sum()
    n_bad_te = (~np.isfinite(X_te)).sum()

    if n_bad_tr > 0 or n_bad_va > 0 or n_bad_te > 0:

        print(
            f"  ⚠️ [{dataset_name}] non-finite values — "
            f"train: {n_bad_tr}, val: {n_bad_va}, test: {n_bad_te} → imputing"
        )

        X_tr = np.where(np.isfinite(X_tr), X_tr, np.nan)
        X_va = np.where(np.isfinite(X_va), X_va, np.nan)
        X_te = np.where(np.isfinite(X_te), X_te, np.nan)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            imputer = SimpleImputer(strategy="median")

            X_tr = imputer.fit_transform(X_tr)
            X_va = imputer.transform(X_va)
            X_te = imputer.transform(X_te)

        X_tr = np.nan_to_num(X_tr, nan=0.0, posinf=0.0, neginf=0.0)
        X_va = np.nan_to_num(X_va, nan=0.0, posinf=0.0, neginf=0.0)
        X_te = np.nan_to_num(X_te, nan=0.0, posinf=0.0, neginf=0.0)

        print(f"  ✓ [{dataset_name}] imputed")

    else:
        print(f"  ✓ [{dataset_name}] no NaN or Inf")

    datasets_svm_undersampled[dataset_name] = {
        "X_train": X_tr.astype(np.float32),
        "y_train": y_tr,

        "X_val": X_va.astype(np.float32),
        "y_val": y_va,

        "X_test": X_te.astype(np.float32),
        "y_test": y_te,

        "nonsep_kept": d.get("nonsep_kept", None),
        "sep_kept": d.get("sep_kept", None),
    }


# ══════════════════════════════════════════════════════════════
# Final summary
# ══════════════════════════════════════════════════════════════

print("\n✅ SVM-ready Catch22 undersampled datasets are ready:")
print(list(datasets_svm_undersampled.keys()))

print(
    f"\n{'Dataset':<18} "
    f"{'Train shape':<18} "
    f"{'Val shape':<18} "
    f"{'Test shape':<18} "
    f"{'Class 0':>8} "
    f"{'Class 1':>8} "
    f"{'Ratio':>8}"
)
print("─" * 105)

for name, d in datasets_svm_undersampled.items():
    counts = np.bincount(d["y_train"].astype(int), minlength=2)
    ratio = counts[0] / counts[1] if counts[1] > 0 else np.inf

    print(
        f"{name:<18} "
        f"{str(d['X_train'].shape):<18} "
        f"{str(d['X_val'].shape):<18} "
        f"{str(d['X_test'].shape):<18} "
        f"{counts[0]:>8} "
        f"{counts[1]:>8} "
        f"{ratio:>7.1f}x"
    )

Pre-extraction value range check:
Dataset            Train shape            Finite?             Min          Max    NaN/Inf  Class 0  Class 1    Ratio
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
tomek_nonsep_8000  (8118, 288, 10)        True            -6.5136      17.8322          0     8000      118    67.8x
tomek_nonsep_2000  (2118, 288, 10)        True            -6.5136      18.1921          0     2000      118    16.9x
tomek_nonsep_500   (618, 288, 10)         True            -6.5136      17.8322          0      500      118     4.2x

Extracting Catch22 features for Tomek + undersampled datasets...

[tomek_nonsep_8000] extracting train...
    tomek_nonsep_8000/train: 8118 samples in 41 chunks
[tomek_nonsep_8000] extracting val...
    tomek_nonsep_8000/val: 1780 samples in 9 chunks
[tomek_nonsep_8000] extracting test...
    tomek_nonsep_8000/test: 3559 samples in 18 chunks
[tomek_nonsep_8000] ✓ train (811

## 3. Classification

### Helper Functions — Metrics and Result I/O

In [2]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
import time
import os

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)


def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    # HSS1 (Barnes & Leka 2008): (TP + TN - N) / P  ==  (TP - FP) / (TP + FN)
    hss1 = (TP - FP) / (TP + FN) if (TP + FN) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    # False Alarm Ratio: of the alarms issued, the fraction that were wrong.
    # nan when no alarm is issued at all -- 0.0 there would read as flawless.
    far = FP / (TP + FP) if (TP + FP) > 0 else float('nan')

    # Frequency bias: alarms issued per event that actually occurred.
    # 1.0 = calibrated, > 1 over-forecasting, < 1 under-forecasting.
    bias = (TP + FP) / (TP + FN) if (TP + FN) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'far': far, 'bias': bias,
        'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'recall': recall, 'precision': precision, 'f1': f1, 'accuracy': accuracy
    }


def train_and_evaluate(model, X_train, y_train, X_test, y_test):
    t0         = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    t0         = time.time()
    y_pred     = model.predict(X_test)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test, y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics


def save_results(metrics_list, filepath):
    with open(filepath, 'w') as f:
        for m in metrics_list:
            line = (f"{m['TP']},{m['TN']},{m['FP']},{m['FN']},"
                    f"{m['tss']:.6f},{m['hss1']:.6f},{m['hss2']:.6f},{m['gss']:.6f},"
                    f"{m['recall']:.6f},{m['f1']:.6f},{m['accuracy']:.6f},"
                    f"{m['train_time']:.4f},{m['infer_time']:.4f},"
                    f"{m['far']:.6f},{m['bias']:.6f}")
            f.write(line + "\n")


def print_results(metrics_list, title):
    keys = ['tss', 'far', 'bias', 'hss1', 'hss2', 'gss',
            'recall', 'precision', 'f1', 'accuracy', 'train_time', 'infer_time']
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print(f"{'─'*55}")
    for i, m in enumerate(metrics_list):
        print(f"  Run {i+1}: TP={m['TP']}  TN={m['TN']}  FP={m['FP']}  FN={m['FN']}")
        print(f"         TSS={m['tss']:.4f}  FAR={m['far']:.4f}  Bias={m['bias']:.2f}")
        print(f"         HSS1={m['hss1']:.4f}  HSS2={m['hss2']:.4f}  GSS={m['gss']:.4f}")
        print(f"         Recall={m['recall']:.4f}  Precision={m['precision']:.4f}  F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}")
        print(f"         Train={m['train_time']:.2f}s  Infer={m['infer_time']:.4f}s")
        print()
    print(f"  ── Average of {len(metrics_list)} runs ──")
    for k in keys:
        avg = np.nanmean([m[k] for m in metrics_list])
        print(f"  {k:<12} : {avg:.4f}")
    print(f"{'─'*55}")

### SVM

In [8]:
import os
import copy
import warnings
import numpy as np

from sklearn.svm import SVC
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ══════════════════════════════════════════════════════════════
# SVM FINAL EXPERIMENTS — TOMEK + NON-SEP UNDERSAMPLING
# Uses fixed best hyperparameters selected previously
# Best model: RBF-SVC, C=0.1, gamma=scale, class_weight=balanced
#
# Input datasets:
# datasets_svm_undersampled
#   tomek_nonsep_8000
#   tomek_nonsep_2000
#   tomek_nonsep_500
# ══════════════════════════════════════════════════════════════

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best SVM hyperparameters
# ---------------------------------------------------------
best_model_class = SVC

best_params = {
    "kernel": "rbf",
    "C": 0.1,
    "gamma": "scale",
    "class_weight": "balanced",
    "cache_size": 2000
}

print("═" * 70)
print("Final SVM experiments on Tomek + undersampled Hybrid datasets")
print("Using fixed best hyperparameters")
print("═" * 70)
print(f"Model  : {best_model_class.__name__}")
print(f"Params : {best_params}")


# ---------------------------------------------------------
# Run final experiments on Tomek + undersampled datasets
# ---------------------------------------------------------
all_undersampled_svm_results = {}

for dataset_key, d in datasets_svm_undersampled.items():

    print("\n" + "─" * 70)
    print(f"Dataset : {dataset_key}")
    print(f"Model   : {best_model_class.__name__}")
    print(f"Params  : {best_params}")

    train_counts = np.bincount(d["y_train"].astype(int), minlength=2)
    val_counts   = np.bincount(d["y_val"].astype(int), minlength=2)
    test_counts  = np.bincount(d["y_test"].astype(int), minlength=2)

    train_ratio = train_counts[0] / train_counts[1] if train_counts[1] > 0 else np.inf

    print(f"Train shape : {d['X_train'].shape}")
    print(f"Val shape   : {d['X_val'].shape}")
    print(f"Test shape  : {d['X_test'].shape}")
    print(f"Train class : Non-SEP={train_counts[0]} | SEP={train_counts[1]} | Ratio={train_ratio:.1f}:1")
    print(f"Val class   : Non-SEP={val_counts[0]} | SEP={val_counts[1]}")
    print(f"Test class  : Non-SEP={test_counts[0]} | SEP={test_counts[1]}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        model = best_model_class(**copy.deepcopy(best_params))

        metrics = train_and_evaluate(
            model,
            d["X_train"], d["y_train"],
            d["X_test"],  d["y_test"]
        )

        metrics_list.append(metrics)

        precision_text = (
            f" | Precision={metrics['precision']:.4f}"
            if "precision" in metrics else ""
        )

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f}"
            f"{precision_text} | "
            f"Train={metrics['train_time']:.2f}s"
        )

    all_undersampled_svm_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"svm_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"SVM | {dataset_key}")


# ---------------------------------------------------------
# Select best undersampled dataset by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("SVM Tomek + undersampling summary")
print("═" * 70)

print(
    f"{'Dataset':<18} "
    f"{'Non-SEP':>8} "
    f"{'SEP':>8} "
    f"{'Ratio':>10} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)
print("─" * 85)

for dataset_key, metrics_list in all_undersampled_svm_results.items():

    d = datasets_svm_undersampled[dataset_key]

    counts = np.bincount(d["y_train"].astype(int), minlength=2)
    ratio = counts[0] / counts[1] if counts[1] > 0 else np.inf

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    print(
        f"{dataset_key:<18} "
        f"{counts[0]:>8} "
        f"{counts[1]:>8} "
        f"{ratio:>9.1f}x "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest SVM Tomek + undersampled dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Model   : {best_model_class.__name__}")
print(f"  Params  : {best_params}")

print("\nAll SVM Tomek + undersampling experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Files: {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

══════════════════════════════════════════════════════════════════════
Final SVM experiments on Tomek + undersampled Hybrid datasets
Using fixed best hyperparameters
══════════════════════════════════════════════════════════════════════
Model  : SVC
Params : {'kernel': 'rbf', 'C': 0.1, 'gamma': 'scale', 'class_weight': 'balanced', 'cache_size': 2000}

──────────────────────────────────────────────────────────────────────
Dataset : tomek_nonsep_8000
Model   : SVC
Params  : {'kernel': 'rbf', 'C': 0.1, 'gamma': 'scale', 'class_weight': 'balanced', 'cache_size': 2000}
Train shape : (8118, 220)
Val shape   : (1780, 220)
Test shape  : (3559, 220)
Train class : Non-SEP=8000 | SEP=118 | Ratio=67.8:1
Val class   : Non-SEP=1763 | SEP=17
Test class  : Non-SEP=3525 | SEP=34
──────────────────────────────────────────────────────────────────────
Run 1: TSS=0.4314 | F1=0.0757 | Recall=0.5588 | Precision=0.0406 | Train=3.55s
Run 2: TSS=0.4314 | F1=0.0757 | Recall=0.5588 | Precision=0.0406 | Train=3.52

### GRU 

In [3]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS — PyTorch
# ══════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import numpy as np
import time
import os
import warnings
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(GRUModel, self).__init__()
        self.gru      = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.bn       = nn.BatchNorm1d(hidden_size)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(hidden_size, 32)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2      = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        out    = self.relu(self.fc1(out))
        out    = self.dropout2(out)
        return self.fc2(out).squeeze()


def train_and_evaluate_gru(params, X_train, y_train, X_val, y_val, X_test, y_test, class_ratio=13):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

    input_size = X_train.shape[2]
    model      = GRUModel(input_size,
                          hidden_size = params["units"],
                          num_layers  = params["layers"],
                          dropout     = params["dropout"]).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size     = params["batch_size"]
    n_samples      = X_tr.shape[0]
    best_val_loss  = float('inf')
    patience_count = 0
    best_state     = None

    t0 = time.time()
    for epoch in range(50):
        model.train()
        perm = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size):
            idx    = perm[i:i + batch_size]
            xb, yb = X_tr[idx], y_tr[idx]
            optimizer.zero_grad()
            loss   = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va), y_va).item()

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= 5:
                break

    train_time = time.time() - t0

    model.load_state_dict(best_state)
    model.eval()
    t0 = time.time()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()
    y_pred     = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics, model


Using device: mps


In [10]:
import os
import numpy as np
import copy

# ══════════════════════════════════════════════════════════════
# GRU FINAL EXPERIMENTS — TOMEK + NON-SEP UNDERSAMPLING
# Uses fixed best hyperparameters selected previously
# Best params: units=64, layers=1, dropout=0.1, lr=0.001, batch_size=64

# Datasets: tomek_nonsep_8000, tomek_nonsep_2000, tomek_nonsep_500
# ══════════════════════════════════════════════════════════════

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best GRU hyperparameters
# ---------------------------------------------------------
best_params = {
    "units": 64,
    "layers": 1,
    "dropout": 0.1,
    "lr": 0.001,
    "batch_size": 64
}

TARGET_VARIANTS = [
    "tomek_nonsep_8000",
    "tomek_nonsep_2000",
    "tomek_nonsep_500",
]

print(f"{'═' * 70}")
print("  Classifier : GRU (PyTorch)")
print("  Experiment : Tomek + Non-SEP undersampling")
print(f"  Best params: {best_params}")
print(f"{'═' * 70}")


# ---------------------------------------------------------
# Run final GRU experiments on undersampled datasets
# ---------------------------------------------------------
all_gru_undersampled_results = {}

for dataset_key in TARGET_VARIANTS:

    if dataset_key not in datasets_undersampled:
        print(f"[{dataset_key}] not found — skipping")
        continue

    d = datasets_undersampled[dataset_key]

    timesteps = d["X_train"].shape[1]

    train_counts = np.bincount(d["y_train"].astype(int), minlength=2)
    val_counts   = np.bincount(d["y_val"].astype(int), minlength=2)
    test_counts  = np.bincount(d["y_test"].astype(int), minlength=2)

    cr = train_counts[0] / train_counts[1]

    print("\n" + "─" * 70)
    print(f"Dataset   : {dataset_key}")
    print(f"Timesteps : {timesteps}")
    print(f"Train     : {d['X_train'].shape} | Non-SEP={train_counts[0]} | SEP={train_counts[1]}")
    print(f"Val       : {d['X_val'].shape} | Non-SEP={val_counts[0]} | SEP={val_counts[1]}")
    print(f"Test      : {d['X_test'].shape} | Non-SEP={test_counts[0]} | SEP={test_counts[1]}")
    print(f"Ratio     : {cr:.1f}:1")
    print(f"Params    : {best_params}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        print(f"Run {run + 1} …", flush=True)

        metrics, _ = train_and_evaluate_gru(
            copy.deepcopy(best_params),
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr
        )

        metrics_list.append(metrics)

        precision_text = (
            f" | Precision={metrics['precision']:.4f}"
            if "precision" in metrics else ""
        )

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f}"
            f"{precision_text} | "
            f"Train={metrics['train_time']:.1f}s"
        )

    all_gru_undersampled_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"gru_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"GRU | {dataset_key}")


# ---------------------------------------------------------
# Select best undersampled dataset by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("GRU Tomek + undersampling summary")
print("═" * 70)

print(
    f"{'Dataset':<18} "
    f"{'Train N':>10} "
    f"{'Non-SEP':>8} "
    f"{'SEP':>8} "
    f"{'Ratio':>10} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)
print("─" * 95)

for dataset_key, metrics_list in all_gru_undersampled_results.items():

    d = datasets_undersampled[dataset_key]

    counts = np.bincount(d["y_train"].astype(int), minlength=2)
    ratio = counts[0] / counts[1]

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    print(
        f"{dataset_key:<18} "
        f"{len(d['y_train']):>10} "
        f"{counts[0]:>8} "
        f"{counts[1]:>8} "
        f"{ratio:>9.1f}x "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest GRU Tomek + undersampled dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Params  : {best_params}")

print("\n✅ All GRU Tomek + undersampling experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Files: {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

══════════════════════════════════════════════════════════════════════
  Classifier : GRU (PyTorch)
  Experiment : Tomek + Non-SEP undersampling
  Best params: {'units': 64, 'layers': 1, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 64}
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
Dataset   : tomek_nonsep_8000
Timesteps : 288
Train     : (8118, 288, 10) | Non-SEP=8000 | SEP=118
Val       : (1780, 288, 10) | Non-SEP=1763 | SEP=17
Test      : (3559, 288, 10) | Non-SEP=3525 | SEP=34
Ratio     : 67.8:1
Params    : {'units': 64, 'layers': 1, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 64}
──────────────────────────────────────────────────────────────────────
Run 1 …
Run 1: TSS=0.6257 | F1=0.0945 | Recall=0.7647 | Precision=0.0504 | Train=73.9s
Run 2 …
Run 2: TSS=0.6684 | F1=0.1071 | Recall=0.7941 | Precision=0.0574 | Train=86.3s

───────────────────────────────────────────────────────
  GRU | tomek

### PatchTST

In [4]:
import os
import time
import copy
import warnings
import numpy as np

import torch
import torch.nn as nn
from transformers import PatchTSTConfig, PatchTSTForClassification

warnings.filterwarnings("ignore")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


def patchtst_label(p):
    return (
        f"patch={p['patch_length']}  "
        f"stride={p['patch_stride']}  "
        f"d={p['d_model']}  "
        f"heads={p['num_attention_heads']}  "
        f"layers={p['num_hidden_layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


def build_patchtst_model(params, seq_len, n_channels):
    config = PatchTSTConfig(
        num_input_channels=n_channels,
        context_length=seq_len,
        num_targets=2,

        patch_length=params["patch_length"],
        patch_stride=params["patch_stride"],

        d_model=params["d_model"],
        num_attention_heads=params["num_attention_heads"],
        num_hidden_layers=params["num_hidden_layers"],
        ffn_dim=params["ffn_dim"],

        attention_dropout=params["dropout"],
        positional_dropout=params["dropout"],
        ff_dropout=params["dropout"],
        head_dropout=params["dropout"],

        pooling_type="mean",
        use_cls_token=True,
        norm_type="layernorm",
        channel_attention=True,
        problem_type="single_label_classification",
        scaling=None,
    )

    return PatchTSTForClassification(config)


def train_and_evaluate_patchtst(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=30,
    patience=3
):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.long).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.long).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    seq_len = X_train.shape[1]
    n_channels = X_train.shape[2]

    model = build_patchtst_model(
        params,
        seq_len=seq_len,
        n_channels=n_channels
    ).to(device)

    class_weights = torch.tensor(
        [1.0, float(class_ratio)],
        dtype=torch.float32
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=1e-4
    )

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()

        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()

            outputs = model(past_values=xb)
            logits = outputs.prediction_logits

            loss = criterion(logits, yb)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_outputs = model(past_values=X_va)
            val_logits = val_outputs.prediction_logits
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    train_time = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    t0 = time.time()

    with torch.no_grad():
        test_outputs = model(past_values=X_te)
        test_logits = test_outputs.prediction_logits
        y_pred = torch.argmax(test_logits, dim=1).cpu().numpy()

    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred.astype(int))
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

Using device: mps


In [5]:
# ══════════════════════════════════════════════════════════════
# PATCHTST FINAL EXPERIMENTS — TOMEK + NON-SEP UNDERSAMPLING
# Uses fixed best hyperparameters selected previously
# Best params: patch=6, stride=6, d=32, heads=4, layers=1,
#              ffn=128, dropout=0.2, lr=0.001, batch_size=128

# Datasets: tomek_nonsep_8000, tomek_nonsep_2000, tomek_nonsep_500
# ══════════════════════════════════════════════════════════════

import os
import copy
import numpy as np

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best PatchTST hyperparameters
# ---------------------------------------------------------
best_params = {
    "patch_length": 6,
    "patch_stride": 6,
    "d_model": 32,
    "num_attention_heads": 4,
    "num_hidden_layers": 1,
    "ffn_dim": 128,
    "dropout": 0.2,
    "lr": 0.001,
    "batch_size": 128,
}

TARGET_VARIANTS = [
    "tomek_nonsep_8000",
    "tomek_nonsep_2000",
    "tomek_nonsep_500",
]


def patchtst_label(p):
    return (
        f"patch={p['patch_length']}  "
        f"stride={p['patch_stride']}  "
        f"d={p['d_model']}  "
        f"heads={p['num_attention_heads']}  "
        f"layers={p['num_hidden_layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


print(f"{'═' * 70}")
print("  Classifier : Hugging Face PatchTST")
print("  Experiment : Tomek + Non-SEP undersampling")
print(f"  Best params: {patchtst_label(best_params)}")
print(f"{'═' * 70}")


# ---------------------------------------------------------
# Run final PatchTST experiments on undersampled datasets
# ---------------------------------------------------------
all_patchtst_undersampled_results = {}

for dataset_key in TARGET_VARIANTS:

    if dataset_key not in datasets_undersampled:
        print(f"[{dataset_key}] not found — skipping")
        continue

    d = datasets_undersampled[dataset_key]

    timesteps = d["X_train"].shape[1]

    train_counts = np.bincount(d["y_train"].astype(int), minlength=2)
    val_counts   = np.bincount(d["y_val"].astype(int), minlength=2)
    test_counts  = np.bincount(d["y_test"].astype(int), minlength=2)

    cr = train_counts[0] / train_counts[1]

    print("\n" + "─" * 70)
    print(f"Dataset   : {dataset_key}")
    print(f"Timesteps : {timesteps}")
    print(f"Train     : {d['X_train'].shape} | Non-SEP={train_counts[0]} | SEP={train_counts[1]}")
    print(f"Val       : {d['X_val'].shape} | Non-SEP={val_counts[0]} | SEP={val_counts[1]}")
    print(f"Test      : {d['X_test'].shape} | Non-SEP={test_counts[0]} | SEP={test_counts[1]}")
    print(f"Ratio     : {cr:.1f}:1")
    print(f"Params    : {patchtst_label(best_params)}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        print(f"Run {run + 1} …", flush=True)

        metrics, _ = train_and_evaluate_patchtst(
            copy.deepcopy(best_params),
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr,
            max_epochs=30,
            patience=3
        )

        metrics_list.append(metrics)

        precision_text = (
            f" | Precision={metrics['precision']:.4f}"
            if "precision" in metrics else ""
        )

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f}"
            f"{precision_text} | "
            f"Train={metrics['train_time']:.1f}s"
        )

    all_patchtst_undersampled_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"patchtst_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"PatchTST | {dataset_key}")


# ---------------------------------------------------------
# Select best undersampled dataset by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("PatchTST Tomek + undersampling summary")
print("═" * 70)

print(
    f"{'Dataset':<18} "
    f"{'Train N':>10} "
    f"{'Non-SEP':>8} "
    f"{'SEP':>8} "
    f"{'Ratio':>10} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)

print("─" * 95)

for dataset_key, metrics_list in all_patchtst_undersampled_results.items():

    d = datasets_undersampled[dataset_key]

    counts = np.bincount(d["y_train"].astype(int), minlength=2)
    ratio = counts[0] / counts[1]

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    print(
        f"{dataset_key:<18} "
        f"{len(d['y_train']):>10} "
        f"{counts[0]:>8} "
        f"{counts[1]:>8} "
        f"{ratio:>9.1f}x "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest PatchTST Tomek + undersampled dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Params  : {patchtst_label(best_params)}")

print("\n✅ All PatchTST Tomek + undersampling experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Files: {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

══════════════════════════════════════════════════════════════════════
  Classifier : Hugging Face PatchTST
  Experiment : Tomek + Non-SEP undersampling
  Best params: patch=6  stride=6  d=32  heads=4  layers=1  drop=0.2  lr=0.001  bs=128
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
Dataset   : tomek_nonsep_8000
Timesteps : 288
Train     : (8118, 288, 10) | Non-SEP=8000 | SEP=118
Val       : (1780, 288, 10) | Non-SEP=1763 | SEP=17
Test      : (3559, 288, 10) | Non-SEP=3525 | SEP=34
Ratio     : 67.8:1
Params    : patch=6  stride=6  d=32  heads=4  layers=1  drop=0.2  lr=0.001  bs=128
──────────────────────────────────────────────────────────────────────
Run 1 …
Run 1: TSS=0.4777 | F1=0.1121 | Recall=0.5588 | Precision=0.0623 | Train=46.1s
Run 2 …
Run 2: TSS=0.4856 | F1=0.1222 | Recall=0.5588 | Precision=0.0686 | Train=45.0s

───────────────────────────────────────────────────────
  PatchTST 

### InceptionTime

In [6]:
import os
import time
import copy
import warnings
import numpy as np

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


# ══════════════════════════════════════════════════════════════
# INCEPTIONTIME MODEL
# ══════════════════════════════════════════════════════════════

class InceptionModule(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels=32,
        kernel_sizes=(9, 19, 39),
        bottleneck_channels=32
    ):
        super(InceptionModule, self).__init__()

        if in_channels > 1:
            self.bottleneck = nn.Conv1d(
                in_channels,
                bottleneck_channels,
                kernel_size=1,
                bias=False
            )
            conv_in_channels = bottleneck_channels
        else:
            self.bottleneck = nn.Identity()
            conv_in_channels = in_channels

        self.conv_list = nn.ModuleList([
            nn.Conv1d(
                conv_in_channels,
                out_channels,
                kernel_size=k,
                padding=k // 2,
                bias=False
            )
            for k in kernel_sizes
        ])

        self.maxpool_conv = nn.Sequential(
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
            nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False)
        )

        total_out_channels = out_channels * (len(kernel_sizes) + 1)

        self.bn = nn.BatchNorm1d(total_out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        x_bottleneck = self.bottleneck(x)

        conv_outputs = [
            conv(x_bottleneck)
            for conv in self.conv_list
        ]

        pool_output = self.maxpool_conv(x)

        out = torch.cat(conv_outputs + [pool_output], dim=1)
        out = self.bn(out)
        out = self.relu(out)

        return out


class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels=32, depth=3, use_residual=True):
        super(InceptionBlock, self).__init__()

        self.use_residual = use_residual
        self.depth = depth

        modules = []
        current_channels = in_channels

        for _ in range(depth):
            module = InceptionModule(
                in_channels=current_channels,
                out_channels=out_channels
            )
            modules.append(module)

            current_channels = out_channels * 4

        self.inception_modules = nn.ModuleList(modules)

        if use_residual:
            self.residual = nn.Sequential(
                nn.Conv1d(in_channels, current_channels, kernel_size=1, bias=False),
                nn.BatchNorm1d(current_channels)
            )
            self.relu = nn.ReLU()

    def forward(self, x):
        residual = x

        out = x
        for module in self.inception_modules:
            out = module(out)

        if self.use_residual:
            residual = self.residual(residual)
            out = self.relu(out + residual)

        return out


class InceptionTimeClassifier(nn.Module):
    def __init__(self, input_channels, out_channels=32, depth=6, dropout=0.3):
        super(InceptionTimeClassifier, self).__init__()

        block_depth = max(1, depth // 2)

        self.block1 = InceptionBlock(
            in_channels=input_channels,
            out_channels=out_channels,
            depth=block_depth,
            use_residual=True
        )

        block1_channels = out_channels * 4

        self.block2 = InceptionBlock(
            in_channels=block1_channels,
            out_channels=out_channels,
            depth=block_depth,
            use_residual=True
        )

        final_channels = out_channels * 4

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(final_channels, 1)

    def forward(self, x):
        # Input shape: (batch, timesteps, features)
        # Conv1d expects: (batch, features, timesteps)
        x = x.permute(0, 2, 1)

        x = self.block1(x)
        x = self.block2(x)

        x = self.gap(x).squeeze(-1)
        x = self.dropout(x)

        return self.fc(x).squeeze()


# ══════════════════════════════════════════════════════════════
# TRAIN AND EVALUATE INCEPTIONTIME
# ══════════════════════════════════════════════════════════════

def train_and_evaluate_inception(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=30,
    patience=3
):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.float32).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    input_channels = X_train.shape[2]

    model = InceptionTimeClassifier(
        input_channels=input_channels,
        out_channels=params["out_channels"],
        depth=params["depth"],
        dropout=params["dropout"]
    ).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=1e-4
    )

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()

        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1

            if patience_count >= patience:
                break

    train_time = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    t0 = time.time()

    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()

    y_pred = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred.astype(int))
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

Using device: mps


In [7]:
# ══════════════════════════════════════════════════════════════
# INCEPTIONTIME FINAL EXPERIMENTS — TOMEK + NON-SEP UNDERSAMPLING
# Uses fixed best hyperparameters selected previously
# Best params: channels=16, depth=8, dropout=0.2, lr=0.001, batch_size=128

# Datasets: tomek_nonsep_8000, tomek_nonsep_2000, tomek_nonsep_500
# ══════════════════════════════════════════════════════════════

import os
import copy
import numpy as np

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best InceptionTime hyperparameters
# ---------------------------------------------------------
best_params = {
    "out_channels": 16,
    "depth": 8,
    "dropout": 0.2,
    "lr": 0.001,
    "batch_size": 128
}

TARGET_VARIANTS = [
    "tomek_nonsep_8000",
    "tomek_nonsep_2000",
    "tomek_nonsep_500",
]


def inception_label(p):
    return (
        f"channels={p['out_channels']}  "
        f"depth={p['depth']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


print(f"{'═' * 70}")
print("  Classifier : InceptionTime")
print("  Experiment : Tomek + Non-SEP undersampling")
print(f"  Best params: {inception_label(best_params)}")
print(f"{'═' * 70}")


# ---------------------------------------------------------
# Run final InceptionTime experiments on undersampled datasets
# ---------------------------------------------------------
all_inception_undersampled_results = {}

for dataset_key in TARGET_VARIANTS:

    if dataset_key not in datasets_undersampled:
        print(f"[{dataset_key}] not found — skipping")
        continue

    d = datasets_undersampled[dataset_key]

    timesteps = d["X_train"].shape[1]

    train_counts = np.bincount(d["y_train"].astype(int), minlength=2)
    val_counts   = np.bincount(d["y_val"].astype(int), minlength=2)
    test_counts  = np.bincount(d["y_test"].astype(int), minlength=2)

    cr = train_counts[0] / train_counts[1]

    print("\n" + "─" * 70)
    print(f"Dataset   : {dataset_key}")
    print(f"Timesteps : {timesteps}")
    print(f"Train     : {d['X_train'].shape} | Non-SEP={train_counts[0]} | SEP={train_counts[1]}")
    print(f"Val       : {d['X_val'].shape} | Non-SEP={val_counts[0]} | SEP={val_counts[1]}")
    print(f"Test      : {d['X_test'].shape} | Non-SEP={test_counts[0]} | SEP={test_counts[1]}")
    print(f"Ratio     : {cr:.1f}:1")
    print(f"Params    : {inception_label(best_params)}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        print(f"Run {run + 1} …", flush=True)

        metrics, _ = train_and_evaluate_inception(
            copy.deepcopy(best_params),
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr,
            max_epochs=30,
            patience=3
        )

        metrics_list.append(metrics)

        precision_text = (
            f" | Precision={metrics['precision']:.4f}"
            if "precision" in metrics else ""
        )

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f}"
            f"{precision_text} | "
            f"Train={metrics['train_time']:.1f}s"
        )

    all_inception_undersampled_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"inceptiontime_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"InceptionTime | {dataset_key}")


# ---------------------------------------------------------
# Select best undersampled dataset by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("InceptionTime Tomek + undersampling summary")
print("═" * 70)

print(
    f"{'Dataset':<18} "
    f"{'Train N':>10} "
    f"{'Non-SEP':>8} "
    f"{'SEP':>8} "
    f"{'Ratio':>10} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)

print("─" * 95)

for dataset_key, metrics_list in all_inception_undersampled_results.items():

    d = datasets_undersampled[dataset_key]

    counts = np.bincount(d["y_train"].astype(int), minlength=2)
    ratio = counts[0] / counts[1]

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    print(
        f"{dataset_key:<18} "
        f"{len(d['y_train']):>10} "
        f"{counts[0]:>8} "
        f"{counts[1]:>8} "
        f"{ratio:>9.1f}x "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest InceptionTime Tomek + undersampled dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Params  : {inception_label(best_params)}")

print("\n✅ All InceptionTime Tomek + undersampling experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Files: {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

══════════════════════════════════════════════════════════════════════
  Classifier : InceptionTime
  Experiment : Tomek + Non-SEP undersampling
  Best params: channels=16  depth=8  drop=0.2  lr=0.001  bs=128
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
Dataset   : tomek_nonsep_8000
Timesteps : 288
Train     : (8118, 288, 10) | Non-SEP=8000 | SEP=118
Val       : (1780, 288, 10) | Non-SEP=1763 | SEP=17
Test      : (3559, 288, 10) | Non-SEP=3525 | SEP=34
Ratio     : 67.8:1
Params    : channels=16  depth=8  drop=0.2  lr=0.001  bs=128
──────────────────────────────────────────────────────────────────────
Run 1 …
Run 1: TSS=0.4793 | F1=0.0913 | Recall=0.5882 | Precision=0.0495 | Train=68.7s
Run 2 …
Run 2: TSS=0.3289 | F1=0.0824 | Recall=0.4118 | Precision=0.0458 | Train=95.3s

───────────────────────────────────────────────────────
  InceptionTime | tomek_nonsep_8000
───────────────────────────